# 03 — Forecasting Model Development

This notebook changes the task from same-record estimation to genuine forecasting.

Default objective:

> Predict available bikes for each station one observation ahead, approximately 30 minutes into the future.

Corrections included:

- Future target created with a station-level shift.
- Lag and rolling features use only past observations.
- Chronological train/validation/test split.
- No target scaling.
- Strong persistence and station-hour baselines.
- XGBoost model evaluated in actual bike units.
- Predictions constrained to valid station capacity.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "Data").exists() or (candidate / "data").exists():
            return candidate.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"
FIGURE_DIR = OUTPUT_DIR / "Figures"
MODEL_DIR = OUTPUT_DIR / "Models"

for directory in (DATA_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /mnt/WorkSpace/Repos/dublin-bikes-fix


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
import joblib

In [3]:
CLEAN_PATH = DATA_DIR / "dataset_cleaned.csv"
FEATURE_PATH = DATA_DIR / "dataset_forecasting_features.csv"

if not CLEAN_PATH.exists():
    raise FileNotFoundError("Run 01_data_cleaning.ipynb first.")

df = pd.read_csv(CLEAN_PATH, parse_dates=["TIME"])
df = df.sort_values(["STATION ID", "TIME"]).reset_index(drop=True)

## Create leakage-safe forecasting features

In [4]:
FORECAST_STEPS = 1

grouped = df.groupby("STATION ID", group_keys=False)

df["TARGET_AVAILABLE_BIKES"] = grouped["AVAILABLE BIKES"].shift(-FORECAST_STEPS)

for lag in [1, 2, 4, 8]:
    df[f"AVAILABLE_BIKES_LAG_{lag}"] = grouped["AVAILABLE BIKES"].shift(lag)

df["ROLLING_MEAN_4"] = grouped["AVAILABLE BIKES"].transform(
    lambda series: series.shift(1).rolling(4, min_periods=1).mean()
)
df["ROLLING_STD_4"] = grouped["AVAILABLE BIKES"].transform(
    lambda series: series.shift(1).rolling(4, min_periods=2).std()
)
df["ROLLING_MEAN_8"] = grouped["AVAILABLE BIKES"].transform(
    lambda series: series.shift(1).rolling(8, min_periods=1).mean()
)

df["HOUR"] = df["TIME"].dt.hour
df["DAY_OF_WEEK"] = df["TIME"].dt.dayofweek
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([5, 6]).astype(int)

df["HOUR_SIN"] = np.sin(2 * np.pi * df["HOUR"] / 24)
df["HOUR_COS"] = np.cos(2 * np.pi * df["HOUR"] / 24)
df["DOW_SIN"] = np.sin(2 * np.pi * df["DAY_OF_WEEK"] / 7)
df["DOW_COS"] = np.cos(2 * np.pi * df["DAY_OF_WEEK"] / 7)

df["MINUTES_SINCE_PREVIOUS"] = (
    grouped["TIME"].diff().dt.total_seconds().div(60)
)

model_df = df.dropna(
    subset=[
        "TARGET_AVAILABLE_BIKES",
        "AVAILABLE_BIKES_LAG_1",
        "AVAILABLE_BIKES_LAG_2",
    ]
).copy()

model_df.to_csv(FEATURE_PATH, index=False)
print(f"Feature dataset shape: {model_df.shape}")
display(model_df.head())

Feature dataset shape: (158648, 28)


,STATION ID,TIME,LAST UPDATED,NAME,BIKE STANDS,AVAILABLE BIKE STANDS,AVAILABLE BIKES,STATUS,ADDRESS,LATITUDE,LONGITUDE,CAPACITY_DIFFERENCE,TARGET_AVAILABLE_BIKES,AVAILABLE_BIKES_LAG_1,AVAILABLE_BIKES_LAG_2,AVAILABLE_BIKES_LAG_4,AVAILABLE_BIKES_LAG_8,ROLLING_MEAN_4,ROLLING_STD_4,ROLLING_MEAN_8,HOUR,DAY_OF_WEEK,IS_WEEKEND,HOUR_SIN,HOUR_COS,DOW_SIN,DOW_COS,MINUTES_SINCE_PREVIOUS
2,2,2021-11-01 01:00:02,2021-11-01 00:54:41,BLESSINGTON STREET,20,9,11,OPEN,Blessington Street,53.3568,-6.26814,0,11.0,11.0,10.0,NaN,NaN,10.500000,0.707107,10.500000,1,0,0,0.258819,0.965926,0.0,1.0,29.983333
3,2,2021-11-01 01:30:03,2021-11-01 01:20:27,BLESSINGTON STREET,20,9,11,OPEN,Blessington Street,53.3568,-6.26814,0,11.0,11.0,11.0,NaN,NaN,10.666667,0.577350,10.666667,1,0,0,0.258819,0.965926,0.0,1.0,30.016667
4,2,2021-11-01 02:00:02,2021-11-01 01:50:46,BLESSINGTON STREET,20,9,11,OPEN,Blessington Street,53.3568,-6.26814,0,11.0,11.0,11.0,10.0,NaN,10.750000,0.500000,10.750000,2,0,0,0.500000,0.866025,0.0,1.0,29.983333
5,2,2021-11-01 02:30:02,2021-11-01 02:21:04,BLESSINGTON STREET,20,9,11,OPEN,Blessington Street,53.3568,-6.26814,0,11.0,11.0,11.0,11.0,NaN,11.000000,0.000000,10.800000,2,0,0,0.500000,0.866025,0.0,1.0,30.000000
6,2,2021-11-01 03:00:03,2021-11-01 02:51:23,BLESSINGTON STREET,20,9,11,OPEN,Blessington Street,53.3568,-6.26814,0,11.0,11.0,11.0,11.0,NaN,11.000000,0.000000,10.833333,3,0,0,0.707107,0.707107,0.0,1.0,30.016667


## Chronological split

In [5]:
unique_times = np.array(sorted(model_df["TIME"].unique()))
train_cutoff = unique_times[int(len(unique_times) * 0.70)]
validation_cutoff = unique_times[int(len(unique_times) * 0.85)]

train_df = model_df[model_df["TIME"] < train_cutoff].copy()
validation_df = model_df[
    (model_df["TIME"] >= train_cutoff)
    & (model_df["TIME"] < validation_cutoff)
].copy()
test_df = model_df[model_df["TIME"] >= validation_cutoff].copy()

assert train_df["TIME"].max() < validation_df["TIME"].min()
assert validation_df["TIME"].max() < test_df["TIME"].min()

split_summary = pd.DataFrame({
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "start": [
        train_df["TIME"].min(),
        validation_df["TIME"].min(),
        test_df["TIME"].min(),
    ],
    "end": [
        train_df["TIME"].max(),
        validation_df["TIME"].max(),
        test_df["TIME"].max(),
    ],
}, index=["train", "validation", "test"])
display(split_summary)

,rows,start,end
train,110807,2021-11-01 01:00:02,2021-11-21 23:30:03
validation,23865,2021-11-22 00:00:03,2021-11-26 11:00:03
test,23976,2021-11-26 11:30:02,2021-11-30 23:00:02


## Feature definition

In [6]:
numeric_features = [
    "BIKE STANDS",
    "LATITUDE",
    "LONGITUDE",
    "AVAILABLE_BIKES_LAG_1",
    "AVAILABLE_BIKES_LAG_2",
    "AVAILABLE_BIKES_LAG_4",
    "AVAILABLE_BIKES_LAG_8",
    "ROLLING_MEAN_4",
    "ROLLING_STD_4",
    "ROLLING_MEAN_8",
    "MINUTES_SINCE_PREVIOUS",
    "HOUR_SIN",
    "HOUR_COS",
    "DOW_SIN",
    "DOW_COS",
    "IS_WEEKEND",
]

categorical_features = ["STATION ID", "STATUS"]
feature_columns = numeric_features + categorical_features
target_column = "TARGET_AVAILABLE_BIKES"

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_validation = validation_df[feature_columns]
y_validation = validation_df[target_column]
X_test = test_df[feature_columns]
y_test = test_df[target_column]

## Baselines

In [7]:
def evaluate_predictions(name, y_true, y_pred):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MedianAE": median_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

results = []

persistence_validation = validation_df["AVAILABLE_BIKES_LAG_1"].to_numpy()
results.append(
    evaluate_predictions(
        "Persistence baseline",
        y_validation,
        persistence_validation
    )
)

station_hour_mean = (
    train_df.groupby(["STATION ID", "HOUR"])[target_column]
    .mean()
)
global_train_mean = y_train.mean()

historical_validation = [
    station_hour_mean.get((station_id, hour), global_train_mean)
    for station_id, hour in zip(validation_df["STATION ID"], validation_df["HOUR"])
]
results.append(
    evaluate_predictions(
        "Station-hour mean baseline",
        y_validation,
        historical_validation
    )
)

display(pd.DataFrame(results).sort_values("MAE"))

,model,MAE,RMSE,MedianAE,R2
0,Persistence baseline,1.687450,3.014144,1.000000,0.868134
1,Station-hour mean baseline,4.950827,6.280909,4.071429,0.427400


## XGBoost pipeline

In [8]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.04,
    max_depth=6,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])

pipeline.fit(X_train, y_train)
validation_pred = pipeline.predict(X_validation)
validation_pred = np.clip(
    validation_pred,
    0,
    validation_df["BIKE STANDS"].to_numpy(),
)

results.append(
    evaluate_predictions("XGBoost", y_validation, validation_pred)
)
validation_results = pd.DataFrame(results).sort_values("MAE")
display(validation_results)

,model,MAE,RMSE,MedianAE,R2
2,XGBoost,1.577663,2.621295,0.854069,0.900267
0,Persistence baseline,1.687450,3.014144,1.000000,0.868134
1,Station-hour mean baseline,4.950827,6.280909,4.071429,0.427400


## Final training and test evaluation

In [9]:
train_validation_df = pd.concat([train_df, validation_df], ignore_index=True)
X_train_validation = train_validation_df[feature_columns]
y_train_validation = train_validation_df[target_column]

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])
final_pipeline.fit(X_train_validation, y_train_validation)

test_pred = final_pipeline.predict(X_test)
raw_invalid_rate = np.mean(
    (test_pred < 0)
    | (test_pred > test_df["BIKE STANDS"].to_numpy())
)
test_pred = np.clip(
    test_pred,
    0,
    test_df["BIKE STANDS"].to_numpy(),
)

test_results = [
    evaluate_predictions(
        "Persistence baseline",
        y_test,
        test_df["AVAILABLE_BIKES_LAG_1"]
    ),
    evaluate_predictions("XGBoost", y_test, test_pred),
]

display(pd.DataFrame(test_results).sort_values("MAE"))
print(f"Raw invalid prediction rate: {raw_invalid_rate:.2%}")

,model,MAE,RMSE,MedianAE,R2
0,Persistence baseline,1.502920,2.766659,1.000000,0.895285
1,XGBoost,1.538793,2.518109,0.896719,0.913254


Raw invalid prediction rate: 1.28%


## Error analysis

In [10]:
prediction_frame = test_df[
    ["TIME", "STATION ID", "BIKE STANDS", target_column]
].copy()
prediction_frame["PREDICTION"] = test_pred
prediction_frame["ABSOLUTE_ERROR"] = (
    prediction_frame[target_column] - prediction_frame["PREDICTION"]
).abs()

station_error = (
    prediction_frame.groupby("STATION ID")["ABSOLUTE_ERROR"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
display(station_error.head(15))

prediction_frame.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

,mean,median,count
STATION ID,,,
9,3.069009,2.053216,216
33,2.909101,2.187868,216
67,2.754714,1.843286,216
34,2.618462,1.635915,216
97,2.583876,1.643047,216
32,2.477038,1.434707,216
40,2.425658,1.630226,216
21,2.423536,1.341550,216
29,2.378028,1.524289,216


## Save complete model bundle

In [11]:
model_bundle = {
    "pipeline": final_pipeline,
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "forecast_steps": FORECAST_STEPS,
    "train_end": str(train_validation_df["TIME"].max()),
    "test_start": str(test_df["TIME"].min()),
    "test_end": str(test_df["TIME"].max()),
    "test_metrics": pd.DataFrame(test_results).to_dict(orient="records"),
}

model_path = MODEL_DIR / "dublin_bikes_xgboost_forecaster.joblib"
joblib.dump(model_bundle, model_path)
print(f"Saved model bundle to {model_path}")

Saved model bundle to /mnt/WorkSpace/Repos/dublin-bikes-fix/Outputs/Models/dublin_bikes_xgboost_forecaster.joblib
